# 02 - Feature Engineering
**Objetivo:** Crear nuevas variables, limpiar categóricas y preparar datos para modelar.
**Nota:** Este notebook valida transformaciones que luego pasarán a `src/feature_engineering.py`

# 1. Importar librerías y cargar datos

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cargar datos limpios (desde 01_EDA)
df = pd.read_csv('UCI_Credit_Card.csv')

# Eliminar ID (no sirve para predecir)
df = df.drop('ID', axis=1)

# Renombrar columnas para legibilidad
df.rename(columns={
    'default.payment.next.month': 'DEFAULT',
    'PAY_0': 'PAY_1',
    'LIMIT_BAL': 'LIMITE_CREDITO',
    'SEX': 'SEXO',
    'EDUCATION': 'EDUCACION',
    'MARRIAGE': 'ESTADO_CIVIL',
    'AGE': 'EDAD'
}, inplace=True)

df.head()

,LIMITE_CREDITO,SEXO,EDUCACION,ESTADO_CIVIL,EDAD,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,DEFAULT
0,20000.0,2,2,1,24,2,2,-1,-1,-2,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,120000.0,2,2,2,26,-1,2,0,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,90000.0,2,2,2,34,0,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,50000.0,2,2,1,37,0,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,50000.0,1,2,1,57,-1,0,-1,0,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


# 2. Limpiar variables categóricas

In [2]:
# 2.1 EDUCACION: agrupar valores raros (0,5,6) en categoría 4 (Otros)
print("Valores originales:")
print(df['EDUCACION'].value_counts().sort_index())

# Mapeo: 1,2,3 se mantienen, el resto → 4
df['EDUCACION'] = df['EDUCACION'].apply(lambda x: x if x in [1,2,3] else 4)

print("\nValores después de limpieza:")
print(df['EDUCACION'].value_counts().sort_index())

Valores originales:
EDUCACION
0       14
1    10585
2    14030
3     4917
4      123
5      280
6       51
Name: count, dtype: int64

Valores después de limpieza:
EDUCACION
1    10585
2    14030
3     4917
4      468
Name: count, dtype: int64


In [3]:
# 2.2 ESTADO_CIVIL: agrupar valor 0 en categoría 3 (Otros)
print("Valores originales:")
print(df['ESTADO_CIVIL'].value_counts().sort_index())

# Mapeo: 1,2 se mantienen, 0 y 3 → 3 (Otros)
# Según documentación: 1=casado, 2=soltero, 3=otros
df['ESTADO_CIVIL'] = df['ESTADO_CIVIL'].apply(lambda x: x if x in [1,2] else 3)

print("\nValores después de limpieza:")
print(df['ESTADO_CIVIL'].value_counts().sort_index())

Valores originales:
ESTADO_CIVIL
0       54
1    13659
2    15964
3      323
Name: count, dtype: int64

Valores después de limpieza:
ESTADO_CIVIL
1    13659
2    15964
3      377
Name: count, dtype: int64


# 3. Crear nuevas variables (Feature Creation)

In [4]:
# 3.1 Máximo atraso en los últimos 6 meses
pagos_atraso = ['PAY_1', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
df['MAX_ATRASO'] = df[pagos_atraso].max(axis=1)

# Ver distribución
df['MAX_ATRASO'].value_counts().sort_index()

MAX_ATRASO
-2     2109
-1     3086
 0    14736
 1     1689
 2     7187
 3      789
 4      218
 5       69
 6       25
 7       67
 8       25
Name: count, dtype: int64

In [5]:
# 3.2 Promedio de atraso (qué tan recurrente es)
df['PROMEDIO_ATRASO'] = df[pagos_atraso].mean(axis=1)

In [6]:
# 3.3 Ratio Pago / Factura (por mes, evitando división por cero)
for i in range(1, 7):
    factura_col = f'BILL_AMT{i}'
    pago_col = f'PAY_AMT{i}'
    ratio_col = f'RATIO_PAGO_{i}'
    
    # Si factura es 0, el ratio es 0 (no hay deuda que pagar)
    df[ratio_col] = df.apply(
        lambda row: row[pago_col] / row[factura_col] if row[factura_col] > 0 else 0,
        axis=1
    )

# Ver las nuevas columnas
df[[f'RATIO_PAGO_{i}' for i in range(1,7)]].head()

,RATIO_PAGO_1,RATIO_PAGO_2,RATIO_PAGO_3,RATIO_PAGO_4,RATIO_PAGO_5,RATIO_PAGO_6
0,0.000000,0.222115,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.579710,0.372856,0.305623,0.000000,0.613309
2,0.051917,0.106937,0.073752,0.069779,0.066899,0.321564
3,0.042562,0.041859,0.024345,0.038850,0.036914,0.033844
4,0.232099,6.469312,0.279057,0.429799,0.035987,0.035492


In [7]:
# 3.4 Ratio Pago Total / Factura Total (últimos 6 meses)
facturas_totales = df[[f'BILL_AMT{i}' for i in range(1,7)]].sum(axis=1)
pagos_totales = df[[f'PAY_AMT{i}' for i in range(1,7)]].sum(axis=1)

df['RATIO_PAGO_TOTAL'] = df.apply(
    lambda row: row['PAY_AMT1'] / row['BILL_AMT1'] if row['BILL_AMT1'] > 0 else 0,
    axis=1
)

# Mejor usar suma total
df['RATIO_PAGO_TOTAL'] = pagos_totales / facturas_totales.replace(0, np.nan)
df['RATIO_PAGO_TOTAL'] = df['RATIO_PAGO_TOTAL'].fillna(0)  # Si no hay factura, ratio = 0

df['RATIO_PAGO_TOTAL'].describe()

count    30000.000000
mean         0.380941
std          7.671004
min       -546.928571
25%          0.040952
50%          0.084932
75%          0.586922
max        797.000000
Name: RATIO_PAGO_TOTAL, dtype: float64

# 4. Escalar variables numéricas

In [8]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Identificar columnas a escalar
columnas_a_escalar = ['LIMITE_CREDITO', 'EDAD'] + \
                      [f'BILL_AMT{i}' for i in range(1,7)] + \
                      [f'PAY_AMT{i}' for i in range(1,7)] + \
                      ['MAX_ATRASO', 'PROMEDIO_ATRASO', 'RATIO_PAGO_TOTAL'] + \
                      [f'RATIO_PAGO_{i}' for i in range(1,7)]

# Opción: StandardScaler (media=0, std=1)
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[columnas_a_escalar] = scaler.fit_transform(df[columnas_a_escalar])

# Verificar
df_scaled[columnas_a_escalar].describe()

,LIMITE_CREDITO,EDAD,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,...,PAY_AMT6,MAX_ATRASO,PROMEDIO_ATRASO,RATIO_PAGO_TOTAL,RATIO_PAGO_1,RATIO_PAGO_2,RATIO_PAGO_3,RATIO_PAGO_4,RATIO_PAGO_5,RATIO_PAGO_6
count,3.000000e+04,3.000000e+04,3.000000e+04,3.000000e+04,3.000000e+04,3.000000e+04,3.000000e+04,3.000000e+04,3.000000e+04,3.000000e+04,...,3.000000e+04,3.000000e+04,3.000000e+04,3.000000e+04,3.000000e+04,3.000000e+04,3.000000e+04,3.000000e+04,3.000000e+04,3.000000e+04
mean,-6.063298e-17,-1.856885e-16,-1.231607e-17,-3.789561e-17,6.252776e-17,5.873820e-17,-2.368476e-17,1.136868e-17,-4.736952e-18,1.373716e-17,...,-1.788199e-17,-1.136868e-17,-2.084259e-17,-5.033011e-19,-1.065814e-18,-9.473903e-19,-1.302662e-18,2.250052e-18,2.368476e-18,8.526513e-18
std,1.000017e+00,1.000017e+00,1.000017e+00,1.000017e+00,1.000017e+00,1.000017e+00,1.000017e+00,1.000017e+00,1.000017e+00,1.000017e+00,...,1.000017e+00,1.000017e+00,1.000017e+00,1.000017e+00,1.000017e+00,1.000017e+00,1.000017e+00,1.000017e+00,1.000017e+00,1.000017e+00
min,-1.213794e+00,-1.571479e+00,-2.944312e+00,-1.671375e+00,-2.945672e+00,-3.315048e+00,-2.000874e+00,-6.355247e+00,-3.419416e-01,-2.569895e-01,...,-2.933821e-01,-1.813007e+00,-1.850576e+00,-7.134902e+01,-2.216692e-02,-3.685470e-02,-3.069554e-02,-2.171318e-02,-6.587230e-02,-5.849642e-02
25%,-9.054983e-01,-8.120745e-01,-6.473120e-01,-6.490466e-01,-6.394814e-01,-6.363293e-01,-6.340600e-01,-6.316338e-01,-2.815661e-01,-2.208358e-01,...,-2.867584e-01,-3.261638e-01,-6.627176e-01,-4.432203e-02,-2.173444e-02,-3.619903e-02,-3.046441e-02,-2.171251e-02,-6.587230e-02,-5.849642e-02
50%,-2.118326e-01,-1.611565e-01,-3.916884e-01,-3.931159e-01,-3.882529e-01,-3.763451e-01,-3.652683e-01,-3.660725e-01,-2.151530e-01,-1.697952e-01,...,-2.090042e-01,-3.261638e-01,1.857528e-01,-3.858860e-02,-2.144750e-02,-3.559447e-02,-2.990088e-02,-2.120249e-02,-6.341188e-02,-5.671124e-02
75%,5.589071e-01,5.982479e-01,2.154919e-01,2.083271e-01,1.896457e-01,1.747667e-01,1.624955e-01,1.733997e-01,-3.970176e-02,-3.998021e-02,...,-6.837436e-02,1.160679e+00,1.857528e-01,2.685240e-02,-1.922587e-02,-3.192559e-02,-2.777657e-02,-2.004413e-02,-5.727088e-02,-5.188166e-02
max,6.416528e+00,4.720729e+00,1.240296e+01,1.313360e+01,2.331820e+01,1.318669e+01,1.458743e+01,1.549528e+01,5.239921e+01,7.284299e+01,...,2.944510e+01,5.621209e+00,6.294740e+00,1.038498e+02,1.564826e+02,1.100300e+02,1.206193e+02,1.196436e+02,1.067783e+02,1.193277e+02


# 5. Separar X (features) y y (target)

In [9]:
# Definir columnas a usar (excluyendo las originales de pago/factura si creamos nuevas)
columnas_para_modelo = [
    'LIMITE_CREDITO', 'SEXO', 'EDUCACION', 'ESTADO_CIVIL', 'EDAD',
    'MAX_ATRASO', 'PROMEDIO_ATRASO', 'RATIO_PAGO_TOTAL'
] + [f'RATIO_PAGO_{i}' for i in range(1,7)]

# Asegurar que todas existan
columnas_para_modelo = [col for col in columnas_para_modelo if col in df_scaled.columns]

X = df_scaled[columnas_para_modelo]
y = df_scaled['DEFAULT']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Columnas: {X.columns.tolist()}")

Features shape: (30000, 14)
Target shape: (30000,)
Columnas: ['LIMITE_CREDITO', 'SEXO', 'EDUCACION', 'ESTADO_CIVIL', 'EDAD', 'MAX_ATRASO', 'PROMEDIO_ATRASO', 'RATIO_PAGO_TOTAL', 'RATIO_PAGO_1', 'RATIO_PAGO_2', 'RATIO_PAGO_3', 'RATIO_PAGO_4', 'RATIO_PAGO_5', 'RATIO_PAGO_6']


# 6. Guardar datos procesados

In [12]:
# Guardar versión escalada para modelar
df_scaled[columnas_para_modelo + ['DEFAULT']].to_csv('credit_card_processed.csv', index=False)
print("✅ Datos procesados guardados en credit_card_processed.csv")

# También guardar el scaler para usarlo en producción
import joblib
joblib.dump(scaler, 'scaler.pkl')
print("✅ Scaler guardado en scaler.pkl")

✅ Datos procesados guardados en credit_card_processed.csv
✅ Scaler guardado en scaler.pkl


# 7. Validación rápida

In [13]:
# Verificar que no hay nulos
print("Nulos por columna:")
print(X.isnull().sum())

# Ver correlación con DEFAULT
corr_con_default = pd.DataFrame({
    'feature': X.columns,
    'correlacion': [X[col].corr(y) for col in X.columns]
}).sort_values('correlacion', ascending=False)

print("\nCorrelación con DEFAULT:")
print(corr_con_default)

Nulos por columna:
LIMITE_CREDITO      0
SEXO                0
EDUCACION           0
ESTADO_CIVIL        0
EDAD                0
MAX_ATRASO          0
PROMEDIO_ATRASO     0
RATIO_PAGO_TOTAL    0
RATIO_PAGO_1        0
RATIO_PAGO_2        0
RATIO_PAGO_3        0
RATIO_PAGO_4        0
RATIO_PAGO_5        0
RATIO_PAGO_6        0
dtype: int64

Correlación con DEFAULT:
             feature  correlacion
5         MAX_ATRASO     0.331036
6    PROMEDIO_ATRASO     0.281955
2          EDUCACION     0.033842
4               EDAD     0.013890
10      RATIO_PAGO_3     0.000231
8       RATIO_PAGO_1    -0.006024
11      RATIO_PAGO_4    -0.007046
13      RATIO_PAGO_6    -0.008406
12      RATIO_PAGO_5    -0.010650
9       RATIO_PAGO_2    -0.011360
7   RATIO_PAGO_TOTAL    -0.011657
3       ESTADO_CIVIL    -0.027575
1               SEXO    -0.039961
0     LIMITE_CREDITO    -0.153520


# 📌 Conclusiones del Feature Engineering

### ✅ Transformaciones aplicadas
1. **EDUCACION**: valores 0,5,6 → categoría 4
2. **ESTADO_CIVIL**: valor 0 → categoría 3
3. **MAX_ATRASO**: peor atraso en 6 meses
4. **PROMEDIO_ATRASO**: qué tan recurrente es el atraso
5. **RATIO_PAGO_1 a RATIO_PAGO_6**: porcentaje pagado vs facturado por mes
6. **RATIO_PAGO_TOTAL**: porcentaje pagado total
7. **Escalado**: StandardScaler aplicado

### 📁 Archivos generados
- `data/credit_card_processed.csv` → datos listos para modelar
- `models/scaler.pkl` → scaler guardado para producción

### 🚀 Próximo paso
Continuar con `03_Modeling_and_Evaluation.ipynb`